# ⛰️ Mountain Entity Detection Demo
This notebook demonstrates the inference capabilities of a custom Named Entity Recognition (NER) model. 

**Project Overview:**
* **Objective:** to automatically detect and extract mountain names (the `MOUNTAIN` entity) from unstructured text.
* **Model Architecture:** the model is built on top of the `microsoft/deberta-v3-base` architecture and fine-tuned for token classification using TensorFlow/Keras.
* **Functionality:** this demo utilizes the custom `MountainDetector` class to process input text, handle subword tokenization, and provide visual HTML-based highlighting of the detected entities.

In [1]:
# !pip install -r requirements.txt

In [2]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent)) 

In [3]:
from src.model_inference import MountainDetector

MODEL_PATH = "../models/deberta_ner_model" 
detector = MountainDetector(MODEL_PATH)

The tokenizer you are loading from '../models/deberta_ner_model' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
TensorFlow and JAX classes are deprecated and will be removed in Transformers v5. We recommend migrating to PyTorch classes or pinning your version of Transformers.
All model checkpoint layers were used when initializing TFDebertaV2ForTokenClassification.

All the layers of TFDebertaV2ForTokenClassification were initialized from the model checkpoint at ../models/deberta_ner_model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFDebertaV2ForTokenClassification for predictions without further training.


In [4]:
from src.model_training import evaluate_model
evaluate_model("../data/processed/test.json", detector.model, detector.tokenizer)

Loading test data from ../data/processed/test.json...


Map:   0%|          | 0/199 [00:00<?, ? examples/s]

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
/Users/aleksandralitvak/Documents/codes/python/Quantum Test Tasks/Natural Language Processing. Named entity recognition/venv_ner/lib/python3.13/site-packages/datasets/arrow_dataset.py:419: FutureWarning: The output of `to_tf_dataset` will change when a passing single element list for `labels` or `columns` in the next datasets version. To return a tuple structure rather than dict, pass a single string.
Old behaviour: columns=['a'], labels=['labels'] -> (tf.Tensor, tf.Tensor)  
             : columns='a', labels='labels' -> (tf.Tensor, tf.Tensor)  
New behaviour: columns=['a'],labels=['labels'] -> ({'a': tf.Tensor}, {'labels': tf.Tensor})  
             : columns='a', labels='labels' -> (tf.Tensor, tf.Tensor) 
  warnings.warn(



--- Evaluation Summary ---
Scenario: all

              correct   incorrect     partial      missed    spurious   precision      recall    f1-score

ent_type          161           0           0           0           2        0.99        1.00        0.99
   exact          161           0           0           0           2        0.99        1.00        0.99
 partial          161           0           0           0           2        0.99        1.00        0.99
  strict          161           0           0           0           2        0.99        1.00        0.99


--- Error Analysis (Spurious / False Positives) ---
Sentence: Huascarán suffered massive rockfall and ice avalanches that altered the topography of the Callejón de Huaylas .
 -> Found: 'Callejón' | Reality: O | Prediction: I-MOUNTAIN
 -> Found: 'Huaylas' | Reality: O | Prediction: I-MOUNTAIN
--------------------------------------------------


Although the model achieves near-perfect precision (0.99), the false positives (spurious entities) reveal an interesting pattern. The model incorrectly tagged parts of "Callejón de Huaylas" as MOUNTAIN, because the surrounding context contained highly mountain-associated vocabulary ("rockfall", "ice avalanches", "topography"). But in reality, this is a valley located next to Mount Huascarán. 

Future Improvement: this highlights the need to introduce more "hard negatives" into the training dataset—specifically, names of valleys, national parks, and rivers that share the same contextual vocabulary as mountains, explicitly labeled as "O".

In [ ]:
test_sentences = [
    ## Basic Cases: Memorization vs. Context (Real vs. Fictional)
    "My friends and I are climbing to Everest.", ## real mountain (seen in training)
    "My friends and I are climbing to Mount Zorblax.", # fictional mountain (zero-shot generalization)
    
    ## Entity Overlap (City/Brand vs. Mountain)
    "After visiting Washington D.C., we decided to hike Mount Washington.", ## real overlap
    "After visiting the city of Eldoria, we decided to hike Mount Eldoriapok.",## fictional overlap
    "After visiting the city of Eldoria, we decided to hike Eldoriapok.", ## fictional overlap without Mount
    
    ## Parts of Speech (Verbs & Metaphors vs. Geography)
    "To mount a winter expedition to K2, you must overcome a mountain of challenges.",
    "She reached the peak of her career shortly before climbing Pikes Peak.",
    "They overcame a mountain of paperwork just to get a permit for Zirkon.", # Fictional mountain with metaphor
    
    ## Brands in Geographical Context
    "We drove our new Ford Everest to the base of Mount Elbrus.",
    
    ## Hard Negatives & Known Limitations (Valleys & Topography)
    "Kyiv is the beautiful capital of Ukraine.",
    "He lives in Mountain View, California, near the headquarters of Google.",
    "The massive rockfall near the Callejón de Huaylas valley blocked the path to the summit."
]

for sentence in test_sentences:
    preds = detector.predict(sentence)
    detector.visualize(sentence, preds)
